# Berth Allocation & Scheduling: Gurobipy model

Bonus part of Phase 2: the same MILP as `pyomo_model.ipynb`, written with **gurobipy** and solved with Gurobi.
The free pip licence of Gurobi is enough for a model of this size.


In [3]:
%pip install gurobipy

In [1]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import ast

# --- Model Initialization ---


In [2]:
model = gp.Model("OperationsResearchModel")

Restricted license - for non-production use only - expires 2025-11-24


# --- Constants and Sets ---


In [4]:
n = 8  
m = 4  
big_M = 10**8
ships_set = range(1, n + 1)
docks_set = range(1, m + 1)
iq_set = gp.tuplelist([(i, q) for i in ships_set for q in ships_set if i != q])
k_set = range(1, 3 + 1) 
bin_set = range(1, 2 + 1)

# --- Data Loading ---


In [12]:
Containers = pd.read_excel('../data/Parameters.xlsx', sheet_name='Containers', index_col=0)
Anchorage = pd.read_excel('../data/Parameters.xlsx', sheet_name='Anchorage', index_col=0)
Unload_Time_processing_df = pd.read_excel('../data/Parameters.xlsx', sheet_name='Unload Time')
Capacity_df = pd.read_excel('../data/Parameters.xlsx', sheet_name='Capacity', index_col=0)
Length_df = pd.read_excel('../data/Parameters.xlsx', sheet_name='Length', index_col=0)
Max_length_df = pd.read_excel('../data/Parameters.xlsx', sheet_name='Max Length', index_col=0)
Lag_df = pd.read_excel('../data/Parameters.xlsx', sheet_name='Lag', index_col=0)
Priority_df = pd.read_excel('../data/Parameters.xlsx', sheet_name='Priority', index_col=0)
Arrival_Time_df = pd.read_excel('../data/Parameters.xlsx', sheet_name='Arrival Time', index_col=0)
Cost_for_Extra_Capacity_df = pd.read_excel('../data/Parameters.xlsx', sheet_name='Cost for Extra Capacity', index_col=0)
Max_Extra_Allowable_Capacity_df = pd.read_excel('../data/Parameters.xlsx', sheet_name='Max Extra Allowable Capacity', index_col=0)

In [13]:
Unload_Time_processing_df['Ship_Berth'] = Unload_Time_processing_df['Ship_Berth'].apply(lambda x: tuple(map(int, ast.literal_eval(x))))
unload_dict = dict(zip(Unload_Time_processing_df['Ship_Berth'], Unload_Time_processing_df['Unload_Time']))

In [14]:
C_param = Capacity_df['Capacity'].to_dict()
CA_param = Containers['Containers'].to_dict()
L_param = Max_length_df['Max_Length'].to_dict()
LS_param = Length_df['Length'].to_dict()
P_param = Priority_df['Priority'].to_dict()
O_param = Lag_df['Lag'].to_dict()
a_param = Arrival_Time_df['Arrival_Time'].to_dict()
CIC_param = Cost_for_Extra_Capacity_df['Cost for Extra Capacity'].to_dict()
ext_param = Max_Extra_Allowable_Capacity_df['Max Extra Allowable Capacity'].to_dict()

# --- Variable Definitions ---

In [15]:
X = model.addVars(ships_set, docks_set, vtype=GRB.BINARY, name="X")
W = model.addVars(ships_set, vtype=GRB.CONTINUOUS, lb=0.0, name="W")
IC = model.addVars(docks_set, vtype=GRB.INTEGER, lb=0, name="IC")
Y = model.addVars(iq_set, docks_set, vtype=GRB.BINARY, name="y") # Using iq_set defined earlier
U = model.addVars(ships_set, k_set, vtype=GRB.CONTINUOUS, lb=0.0, name="u")
F = model.addVars(ships_set, bin_set, vtype=GRB.BINARY, name="f")
START = model.addVars(ships_set, vtype=GRB.CONTINUOUS, lb=0.0, name="start")

# --- Objective Function ---

In [16]:
obj_expr = (
    25 * gp.quicksum(U[i, 1] for i in ships_set) +
    10 * gp.quicksum(U[i, 2] for i in ships_set) +
    5 * gp.quicksum(U[i, 3] for i in ships_set) +
    5 * gp.quicksum(unload_dict[(i, j)] * X[i, j] for i in ships_set for j in docks_set if (i,j) in unload_dict) +
    gp.quicksum(CIC_param[j] * IC[j] for j in docks_set)
)
model.setObjective(obj_expr, GRB.MINIMIZE)

# --- Constraints ---


In [17]:
model.addConstrs((gp.quicksum(X[i, j] for j in docks_set) == 1
                  for i in ships_set), name="c1_one_pier_to_ship")

for j in docks_set:
    if j == 3:
        model.addConstr(gp.quicksum(X[i, j] for i in ships_set) == 3, name=f"c2_ships_at_pier_{j}")
    else:
        model.addConstr(gp.quicksum(X[i, j] for i in ships_set) <= 2, name=f"c2_ships_at_pier_{j}")

model.addConstrs((gp.quicksum(X[i, j] * CA_param[i] for i in ships_set) <= C_param[j] + IC[j]
                  for j in docks_set), name="c3_container_capacity")

model.addConstrs((IC[j] <= ext_param[j]
                  for j in docks_set), name="c4_extra_capacity_ceiling")

model.addConstrs((LS_param[i] * X[i, j] <= L_param[j]
                  for i in ships_set for j in docks_set), name="c5_ship_length")

model.addConstrs((W[i] <= 80 * (1 - P_param[i])
                  for i in ships_set), name="c6_max_wait")

model.addConstrs((W[i] == START[i] - a_param[i]
                  for i in ships_set), name="c7_wait_time_def")

for i_ship, q_ship in iq_set:
    for j_dock in docks_set:
        if (i_ship, j_dock) in unload_dict:
             model.addConstr(
                 a_param[i_ship] + W[i_ship] + unload_dict[(i_ship, j_dock)] + O_param[j_dock] <=
                 START[q_ship] + big_M * (1 - Y[i_ship, q_ship, j_dock]),
                 name=f"c8_not_unloading_{i_ship}_{q_ship}_{j_dock}"
             )

for i_ship, q_ship in iq_set:
    for j_dock in docks_set:
        model.addConstr(Y[i_ship, q_ship, j_dock] + Y[q_ship, i_ship, j_dock] <= 1,
                        name=f"c9_y_sum_le_1_{i_ship}_{q_ship}_{j_dock}")
        model.addConstr(Y[i_ship, q_ship, j_dock] + Y[q_ship, i_ship, j_dock] >=
                        X[i_ship, j_dock] + X[q_ship, j_dock] - 1,
                        name=f"c10_y_sum_ge_X_{i_ship}_{q_ship}_{j_dock}")
        model.addConstr(Y[i_ship, q_ship, j_dock] + Y[q_ship, i_ship, j_dock] <= X[i_ship, j_dock],
                        name=f"c11_y_sum_le_Xi_{i_ship}_{q_ship}_{j_dock}")

model.addConstrs((START[i] >= a_param[i]
                  for i in ships_set), name="c12_start_after_arrival")

for i_ship in ships_set:
    j_dock = 3
    if j_dock in docks_set:
        sum_y_ijk = gp.quicksum(Y[i_ship, k_ship, j_dock] for k_ship in ships_set if k_ship != i_ship) 
        model.addConstr(W[i_ship] <= 80 * (2 - sum_y_ijk),
                        name=f"c13_wait_zero_dock3_{i_ship}")

for i_ship, q_ship in iq_set: 
    for j_dock in docks_set:
        if j_dock != 3:
            model.addConstr(W[i_ship] <= 80 * (1 - Y[i_ship, q_ship, j_dock]),
                            name=f"c13_wait_zero_otherdocks_{i_ship}_{q_ship}_{j_dock}")

model.addConstrs((W[i] == gp.quicksum(U[i, k] for k in k_set)
                  for i in ships_set), name="c14_W_equals_sum_U")

for i in ships_set:
    for b_idx in bin_set:
        if b_idx == 1:
            model.addConstr(10 * F[i, b_idx] <= U[i, b_idx], name=f"c15_F_U_rel_{i}_{b_idx}")
        elif b_idx == 2:
            model.addConstr(30 * F[i, b_idx] <= U[i, b_idx], name=f"c15_F_U_rel_{i}_{b_idx}")

for i in ships_set:
    for k_idx in k_set:
        if k_idx == 1:
            model.addConstr(U[i, k_idx] <= 10, name=f"c16_U_bound_{i}_{k_idx}")
        elif k_idx == 2:
            model.addConstr(U[i, k_idx] <= 30 * F[i, 1], name=f"c16_U_bound_{i}_{k_idx}")
        elif k_idx == 3:
            model.addConstr(U[i, k_idx] <= 30 * F[i, 2], name=f"c16_U_bound_{i}_{k_idx}")

model.addConstrs((F[i, 1] >= F[i, 2]
                  for i in ships_set), name="c17_F_ordering")

{1: <gurobi.Constr *Awaiting Model Update*>,
 2: <gurobi.Constr *Awaiting Model Update*>,
 3: <gurobi.Constr *Awaiting Model Update*>,
 4: <gurobi.Constr *Awaiting Model Update*>,
 5: <gurobi.Constr *Awaiting Model Update*>,
 6: <gurobi.Constr *Awaiting Model Update*>,
 7: <gurobi.Constr *Awaiting Model Update*>,
 8: <gurobi.Constr *Awaiting Model Update*>}

# --- Solve ---


In [19]:
model.optimize()

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i7-6600U CPU @ 2.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 1204 rows, 316 columns and 3376 nonzeros
Model fingerprint: 0x7a5e2764
Variable types: 40 continuous, 276 integer (272 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+08]
  Objective range  [5e+00, 5e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+08]
Presolved: 93 rows, 51 columns, 301 nonzeros

Continuing optimization...


Cutting planes:
  Cover: 2
  Implied bound: 2
  MIR: 3
  Flow cover: 2
  RLT: 3
  Relax-and-lift: 4

Explored 8 nodes (342 simplex iterations) in 0.02 seconds (0.00 work units)
Thread count was 4 (of 4 available processors)

Solution count 2: 1803.27 1855.29 

Optimal solution found (tolerance 1.00e-04)
Best objective 1.803270601388e+03, best bound 1.803270601388e+03, 

# --- Results ---

In [20]:
if model.Status == GRB.OPTIMAL:
    print("\nOptimal solution found!")
    print(f"Objective value: {model.ObjVal:.4f}")

    print("\n--- Variable Values ---")
    for i in ships_set:
        for j in docks_set:
            if X[i, j].X > 0.5:
                print(f"X[{i}, {j}] = {X[i, j].X:.0f} (Ship {i} to Dock {j})")

    print("\nWaiting Times (W):")
    for i in ships_set:
        if W[i].X > 1e-6:
            print(f"W[{i}] = {W[i].X:.4f}")

    print("\nStart Times (start):")
    for i in ships_set:
        if START[i].X > 1e-6:
            print(f"start[{i}] = {START[i].X:.4f}")
    
    print("\nIncreased Capacities (IC):")
    for j in docks_set:
        if IC[j].X > 0.5:
             print(f"IC[{j}] = {IC[j].X:.0f}")

    print("\nPrecedence Variables (y):")
    for i_ship, q_ship in iq_set:
        for j_dock in docks_set:
            if Y[i_ship, q_ship, j_dock].X > 0.5:
                print(f"y[{i_ship},{q_ship},{j_dock}] = {Y[i_ship, q_ship, j_dock].X:.0f} (Ship {i_ship} precedes {q_ship} at Dock {j_dock})")

    print("\nWaiting Time Components (u):")
    for i in ships_set:
        for k_idx in k_set:
            if U[i, k_idx].X > 1e-6:
                print(f"u[{i},{k_idx}] = {U[i, k_idx].X:.4f}")
    
    print("\nBinary Indicators for U (f):")
    for i in ships_set:
        for b_idx in bin_set:
            if F[i, b_idx].X > 0.5:
                print(f"f[{i},{b_idx}] = {F[i, b_idx].X:.0f}")

elif model.Status == GRB.INFEASIBLE:
    print("\nModel is infeasible.")
    print("Computing IIS to help identify conflicting constraints...")
    model.computeIIS()
    model.write("model_iis.ilp")
    print("IIS written to model_iis.ilp")
elif model.Status == GRB.UNBOUNDED:
    print("\nModel is unbounded.")
else:
    print(f"\nOptimization ended with status: {model.Status}")


Optimal solution found!
Objective value: 1803.2706

--- Variable Values ---
X[1, 2] = 1 (Ship 1 to Dock 2)
X[2, 4] = 1 (Ship 2 to Dock 4)
X[3, 3] = 1 (Ship 3 to Dock 3)
X[4, 1] = 1 (Ship 4 to Dock 1)
X[5, 3] = 1 (Ship 5 to Dock 3)
X[6, 2] = 1 (Ship 6 to Dock 2)
X[7, 3] = 1 (Ship 7 to Dock 3)
X[8, 4] = 1 (Ship 8 to Dock 4)

Waiting Times (W):
W[3] = 19.2671
W[5] = 39.3789
W[6] = 20.6958
W[8] = 8.2408

Start Times (start):
start[1] = 4.7655
start[2] = 1.0046
start[3] = 22.9201
start[4] = 6.0542
start[5] = 41.2481
start[6] = 22.8138
start[7] = 4.6230
start[8] = 14.8420

Increased Capacities (IC):
IC[4] = 1

Precedence Variables (y):
y[1,6,2] = 1 (Ship 1 precedes 6 at Dock 2)
y[2,8,4] = 1 (Ship 2 precedes 8 at Dock 4)
y[3,5,3] = 1 (Ship 3 precedes 5 at Dock 3)
y[7,3,3] = 1 (Ship 7 precedes 3 at Dock 3)
y[7,5,3] = 1 (Ship 7 precedes 5 at Dock 3)

Waiting Time Components (u):
u[3,1] = 10.0000
u[3,2] = 9.2671
u[5,1] = 10.0000
u[5,2] = 29.3789
u[6,1] = 10.0000
u[6,2] = 10.6958
u[8,1] = 8.2408